# What does the model actually output next? (diagnostic)
The grid warned that the model puts almost no probability on the digit tokens `1`-`4` right after the prompt (`digit_mass` ~ 0.03). Here we look at the **first 20 dev questions, one at a time**, in exactly one setting:

**Qwen3-0.6B, chat template OFF (raw prompt ending in `පිළිතුර:`), no calibration, one option order, nosubject prompt.**
(Set `CHAT = True` in section 2 to repeat it with the chat template.)

Uses **dev data only** (public train). Runtime: T4 GPU. Secrets: `HF_TOKEN`.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Sinhala MMLU"
!pip -q install -r requirements-colab.txt
import os, sys
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
sys.path.insert(0, 'src')

## 2. Load the model and the first 20 dev questions

In [ ]:
MODEL = 'Qwen/Qwen3-0.6B'
DTYPE = 'fp16'
PROMPT = 'nosubject'
N = 20
CHAT = False                    # False = raw prompt, True = chat template

import debug_next_token as D
from data_utils import load_jsonl
rows = load_jsonl('dev/dev_easy_full.jsonl')[:N]
tok, model = D.load(MODEL, DTYPE)
print(MODEL, '| prompt:', PROMPT, '| chat template:', 'ON' if CHAT else 'OFF', '| calibration: none | questions:', len(rows))

## 3. Question by question
For each question: the **exact text fed to the model** (`repr`, so spaces / newlines / special tokens are visible), the **last 12 input tokens**, the **top 15 next tokens** (digits marked), the **probability of the digit tokens at the position the scorer reads**, the **candidates `'1'` vs `' 1'`** (one token or two, and their probability), and the **greedy continuation** (what the model writes if left alone).

In [ ]:
for i, row in enumerate(rows):
    print('#' * 100)
    print(f'QUESTION {i}  |  {row["subject"]}  |  gold answer: {row["answer"]}')
    D.show_question(tok, model, row, prompt_mode=PROMPT, chat=CHAT)

## 4. Summary over the same 20 questions
The most common top-1 next token and the mean probability on the digit tokens. If the most common top-1 token is a space, a newline or a word, that is why the digit mass is low.

In [ ]:
D.summary(tok, model, rows, prompt_mode=PROMPT, chat=CHAT)

## How to read the results
| You see | Meaning | Likely fix |
|---|---|---|
| Top-1 is a space token (or `Ġ` / `▁`), and `' 1'` is **two tokens** | the answer is written as space + digit, but the scorer reads the position **before** the space | score the two-token answer (probability of the space times probability of the digit after it) |
| Top-1 is a Sinhala word (e.g. `පිළිතුර`, or the question again) | the model does not start with the digit | change the instruction / prompt ending, or constrain the answer format |
| Top-1 is a thinking tag (`<think>`) | thinking is on in the template | check `enable_thinking=False` |
| Total probability of all 8 candidates is high but the digit mass is low | the digit is there, the scorer reads the wrong token | fix the scorer, not the prompt |
| All of them are tiny | the model cannot follow the instruction (too small / wrong format) | choose another model or format; not a code bug |